# 🗄️ Lab 02: Vector DB Deep Dive - 向量資料庫深入探討

## 學習目標
在本實驗中，您將學習：
1. **資源估算 (Resource Estimation)** - 計算向量資料庫的記憶體需求
2. **ChromaDB 進階操作** - Metadata filtering、持久化存儲
3. **FAISS 向量搜索** - Facebook AI 高效向量索引
4. **Qdrant 向量資料庫** - 進階過濾與效能比較
5. **效能基準測試** - 比較不同 Vector DB 的表現

## 技術棧 (全部本地執行)
- **Vector DBs**: ChromaDB, FAISS, Qdrant
- **Embedding Model**: `sentence-transformers/all-MiniLM-L6-v2`
- **Framework**: LangChain

---

## 📦 Part 1: 環境設置

In [ ]:
# 安裝必要套件
!pip install --quiet langchain langchain-community langchain-text-splitters
!pip install --quiet chromadb faiss-cpu qdrant-client
!pip install --quiet sentence-transformers langchain-huggingface
!pip install --quiet requests numpy pandas matplotlib


In [ ]:
# 準備資料 (使用 Lab 01 相同的資料)
import os

# 設定本地 HuggingFace cache 目錄
os.environ["HF_HOME"] = "./hf_cache"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
import requests
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

# 建立資料目錄並下載範例文件
os.makedirs('data', exist_ok=True)
url = "https://www.gutenberg.org/files/11/11-0.txt"
output_path = "data/alice_in_wonderland.txt"

if not os.path.exists(output_path):
    response = requests.get(url)
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(response.text)
    print(f"✅ 已下載: {output_path}")

# 載入並分割文件
loader = TextLoader(output_path, encoding='utf-8')
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True
)
chunks = text_splitter.split_documents(documents)

# 初始化 embedding 模型
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    cache_folder="./hf_cache",
    model_kwargs={"device": "cuda"},
)

print(f"📚 載入了 {len(chunks)} 個 chunks")
print(f"📐 Embedding 維度: {len(embeddings.embed_query('test'))}")



In [ ]:
---
## 📊 Part 2: 資源估算 (Resource Estimation)

了解向量資料庫的記憶體需求對於系統設計至關重要。

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 15.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 105.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 10.1 MB/s eta 0:00:00
   ━━

In [ ]:
import numpy as np

def estimate_memory(num_vectors, embedding_dim, dtype='float32'):
    """估算向量存儲所需的記憶體"""
    bytes_per_element = {'float16': 2, 'float32': 4, 'float64': 8}
    
    # 向量存儲
    vector_bytes = num_vectors * embedding_dim * bytes_per_element[dtype]
    
    # 索引開銷 (約 10-30% 額外)
    index_overhead = vector_bytes * 0.2
    
    # 元數據開銷 (預估每條記錄 200 bytes)
    metadata_bytes = num_vectors * 200
    
    total_bytes = vector_bytes + index_overhead + metadata_bytes
    
    return {
        'vectors_mb': vector_bytes / (1024**2),
        'index_mb': index_overhead / (1024**2),
        'metadata_mb': metadata_bytes / (1024**2),
        'total_mb': total_bytes / (1024**2),
        'total_gb': total_bytes / (1024**3)
    }

# 當前資料估算
num_chunks = len(chunks)
embedding_dim = 384  # all-MiniLM-L6-v2 的維度

print("📊 資源估算報告")
print("=" * 50)
estimate = estimate_memory(num_chunks, embedding_dim)
print(f"📄 Chunks 數量: {num_chunks}")
print(f"📐 Embedding 維度: {embedding_dim}")
print(f"\n💾 記憶體估算:")
print(f"   向量存儲: {estimate['vectors_mb']:.2f} MB")
print(f"   索引開銷: {estimate['index_mb']:.2f} MB")
print(f"   元數據: {estimate['metadata_mb']:.2f} MB")
print(f"   總計: {estimate['total_mb']:.2f} MB")

# 大規模場景估算
print("\n📈 大規模場景估算:")
print("-" * 50)
for num_docs in [10000, 100000, 1000000]:
    est = estimate_memory(num_docs, embedding_dim)
    print(f"   {num_docs:>10,} 文檔: {est['total_gb']:.2f} GB")



---
## 🔵 Part 3: ChromaDB - 進階操作

ChromaDB 是一個輕量級、本地優先的向量資料庫，非常適合開發和小規模部署。

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
import time

# 為 chunks 添加豐富的 metadata
updated_chunks = []
chapters = ['Introduction', 'Rabbit Hole', 'Pool of Tears', 'Caucus Race', 
            'Caterpillar', 'Pig and Pepper', 'Mad Tea-Party', 'Croquet', 
            'Mock Turtle', 'Lobster Quadrille', 'Trial']

for i, chunk in enumerate(chunks):
    new_metadata = chunk.metadata.copy()
    new_metadata['chunk_id'] = i
    new_metadata['chapter'] = chapters[i % len(chapters)]
    new_metadata['word_count'] = len(chunk.page_content.split())
    new_metadata['has_dialogue'] = '"' in chunk.page_content
    updated_chunks.append(Document(page_content=chunk.page_content, metadata=new_metadata))

print(f"📝 已為 {len(updated_chunks)} 個 chunks 添加 metadata")
print(f"📋 Metadata 範例: {updated_chunks[0].metadata}")



In [ ]:
# 建立 ChromaDB 向量存儲
print("⏳ 建立 ChromaDB 向量存儲...")
start_time = time.time()

chroma_db = Chroma.from_documents(
    documents=updated_chunks,
    embedding=embeddings,
    persist_directory="./chroma_lab02",
    collection_name="alice_metadata"
)

chroma_time = time.time() - start_time
print(f"✅ ChromaDB 建立完成！耗時: {chroma_time:.2f} 秒")
print(f"📊 共存儲 {chroma_db._collection.count()} 個向量")

Directory 'data' created.


### 3.1 Metadata Filtering 示範



In [ ]:
query = "What happened at the tea party?"

# 1. 基本搜索 (無過濾)
print("🔍 基本搜索 (無過濾)")
print("=" * 50)
results = chroma_db.similarity_search(query, k=3)
for i, doc in enumerate(results):
    print(f"   {i+1}. Chapter: {doc.metadata['chapter']}, Words: {doc.metadata['word_count']}")

# 2. 按章節過濾
print("\n🔍 按章節過濾 (只搜索 Mad Tea-Party)")
print("=" * 50)
results_filtered = chroma_db.similarity_search(
    query, k=3, 
    filter={"chapter": "Mad Tea-Party"}
)
for i, doc in enumerate(results_filtered):
    print(f"   {i+1}. Chapter: {doc.metadata['chapter']}")
    print(f"      內容: {doc.page_content[:100]}...")

# 3. 複合條件過濾
print("\n🔍 複合條件過濾 (字數 > 100 且有對話)")
print("=" * 50)
results_complex = chroma_db.similarity_search(
    query, k=3,
    filter={"$and": [
        {"word_count": {"$gt": 100}},
        {"has_dialogue": True}
    ]}
)
for i, doc in enumerate(results_complex):
    print(f"   {i+1}. Words: {doc.metadata['word_count']}, Dialogue: {doc.metadata['has_dialogue']}")

Successfully downloaded 'data/alice_in_wonderland.txt'.


---
## 🟠 Part 4: FAISS - 高效向量搜索

FAISS (Facebook AI Similarity Search) 是 Meta 開發的高效向量相似度搜索庫，特別適合大規模數據。


In [ ]:
from langchain_community.vectorstores import FAISS

# 建立 FAISS 向量存儲
print("⏳ 建立 FAISS 向量存儲...")
start_time = time.time()

faiss_db = FAISS.from_documents(
    documents=updated_chunks,
    embedding=embeddings
)

faiss_time = time.time() - start_time
print(f"✅ FAISS 建立完成！耗時: {faiss_time:.2f} 秒")

# 測試 FAISS 搜索
query = "Who is the Queen of Hearts?"
results = faiss_db.similarity_search_with_score(query, k=3)

print(f"\n🔍 FAISS 搜索: '{query}'")
print("=" * 50)
for i, (doc, score) in enumerate(results):
    print(f"   {i+1}. 距離分數: {score:.4f}")
    print(f"      內容: {doc.page_content[:100]}...")

---
## 🟢 Part 5: Qdrant - 進階向量資料庫

Qdrant 提供了更豐富的過濾功能和效能優化選項。



In [ ]:
from langchain_community.vectorstores import Qdrant

# 建立 Qdrant 向量存儲 (本地模式)
print("⏳ 建立 Qdrant 向量存儲...")
start_time = time.time()

qdrant_db = Qdrant.from_documents(
    documents=updated_chunks,
    embedding=embeddings,
    location=":memory:",  # 使用記憶體存儲
    collection_name="alice_qdrant"
)

qdrant_time = time.time() - start_time
print(f"✅ Qdrant 建立完成！耗時: {qdrant_time:.2f} 秒")

# 測試 Qdrant 搜索
results = qdrant_db.similarity_search_with_score(query, k=3)
print(f"\n🔍 Qdrant 搜索: '{query}'")
print("=" * 50)
for i, (doc, score) in enumerate(results):
    print(f"   {i+1}. 相似度分數: {score:.4f}")
    print(f"      內容: {doc.page_content[:100]}...")

ModuleNotFoundError: No module named 'langchain_community'

---
## 📈 Part 6: 效能比較

比較三個 Vector DB 的建立時間和查詢效能。



In [ ]:
# 效能比較：查詢延遲
import pandas as pd

def benchmark_search(db, db_name, query, n_iterations=10):
    """測量查詢延遲"""
    times = []
    for _ in range(n_iterations):
        start = time.time()
        db.similarity_search(query, k=5)
        times.append(time.time() - start)
    return {
        'db': db_name,
        'avg_ms': np.mean(times) * 1000,
        'std_ms': np.std(times) * 1000,
        'min_ms': np.min(times) * 1000,
        'max_ms': np.max(times) * 1000
    }

# 執行基準測試
print("📊 執行查詢效能基準測試...")
print("=" * 60)

results = []
results.append(benchmark_search(chroma_db, "ChromaDB", query))
results.append(benchmark_search(faiss_db, "FAISS", query))
results.append(benchmark_search(qdrant_db, "Qdrant", query))

# 顯示結果
df = pd.DataFrame(results)
print("\n📈 查詢延遲比較 (毫秒):")
print(df.to_string(index=False))

# 建立時間比較
print(f"\n⏱️ 建立時間比較:")
print(f"   ChromaDB: {chroma_time:.2f} 秒")
print(f"   FAISS:    {faiss_time:.2f} 秒")
print(f"   Qdrant:   {qdrant_time:.2f} 秒")

SyntaxError: invalid syntax (ipython-input-3686645823.py, line 1)

---
## 🎯 Lab 02 總結

### Vector DB 特性比較

| 特性 | ChromaDB | FAISS | Qdrant |
|------|----------|-------|--------|
| 安裝複雜度 | 簡單 | 中等 | 中等 |
| Metadata 過濾 | ✅ 豐富 | ❌ 有限 | ✅ 豐富 |
| 持久化 | ✅ | ✅ | ✅ |
| 雲端支援 | ❌ | ❌ | ✅ |
| 適用場景 | 開發/小規模 | 大規模/高效能 | 生產環境 |

### 學習重點
1. 資源估算對系統設計很重要
2. 不同 Vector DB 有各自的優缺點
3. Metadata filtering 能提升檢索精確度
4. 選擇 Vector DB 要考慮實際需求

### 下一步
- **Lab 03**: Hybrid Search 混合搜索策略



In [7]:
get_ipython().system('pip install --quiet langchain-community')
print("Installed langchain-community.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
google-adk 1.21.0 requires opentelemetry-api<=1.37.0,>=1.37.0, but you have opentelemetry-api 1.39.1 which is incompatible.
google-adk 1.21.0 requires opentelemetry-sdk<=1.37.0,>=1.37.0, but you have opentelemetry-sdk 1.39.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.37.0 requires opentelemetry-exporter-otlp-proto-common==1.37.0, but you have opentelemetry-exporter-otlp-proto-common 1.39.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.37.0 requires ope

**Reasoning**:
Since `langchain-community` has been successfully installed, the next step is to re-run the code cell that loads the 'alice_in_wonderland.txt' document using `TextLoader` from `langchain_community.document_loaders`, which was the original intent after the lab introduction.



In [8]:
from langchain_community.document_loaders import TextLoader

# Load the document
loader = TextLoader('data/alice_in_wonderland.txt', encoding='utf-8')
documents = loader.load()

print(f"Loaded {len(documents)} document(s).")
print(f"First 200 characters of the document:\n{documents[0].page_content[:200]}")

Loaded 1 document(s).
First 200 characters of the document:
*** START OF THE PROJECT GUTENBERG EBOOK 11 ***

[Illustration]




Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Ho


**Reasoning**:
Following the instructions, the next step is to split the loaded `documents` into smaller, manageable chunks using `RecursiveCharacterTextSplitter` from `langchain.text_splitter` with specified `chunk_size` and `chunk_overlap`.



In [9]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    add_start_index=True,
)

# Split the documents into chunks
chunks = text_splitter.split_documents(documents)

print(f"Split {len(documents)} document(s) into {len(chunks)} chunks.")
print(f"First chunk (first 200 chars):\n{chunks[0].page_content[:200]}")
print(f"First chunk metadata:\n{chunks[0].metadata}")

SyntaxError: unterminated f-string literal (detected at line 15) (ipython-input-3314092846.py, line 15)

**Reasoning**:
The previous code failed due to an `SyntaxError: unterminated f-string literal`. This occurs because a newline character was present directly within a single-line f-string. To resolve this, the f-strings for printing the chunk content and metadata will be converted to triple-quoted f-strings, allowing them to span multiple lines correctly.



In [10]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    add_start_index=True,
)

# Split the documents into chunks
chunks = text_splitter.split_documents(documents)

print(f"Split {len(documents)} document(s) into {len(chunks)} chunks.")
print(f"First chunk (first 200 chars):\n{chunks[0].page_content[:200]}")
print(f"First chunk metadata:\n{chunks[0].metadata}")

ModuleNotFoundError: No module named 'langchain.text_splitter'

**Reasoning**:
The previous code failed with a `ModuleNotFoundError: No module named 'langchain.text_splitter'`. This indicates that the `RecursiveCharacterTextSplitter` class, which was previously imported from `langchain.text_splitter`, is now located in a different module or requires a separate installation. The correct package to import it from is `langchain_text_splitters`. Therefore, the immediate next step is to install this missing package.



In [11]:
get_ipython().system('pip install --quiet langchain-text-splitters')
print("Installed langchain-text-splitters.")

Installed langchain-text-splitters.


**Reasoning**:
Since `langchain-text-splitters` has been successfully installed, the next step is to re-run the code cell that splits the document into chunks using `RecursiveCharacterTextSplitter`, this time importing it from `langchain_text_splitters` as previously identified.



In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    add_start_index=True,
)

# Split the documents into chunks
chunks = text_splitter.split_documents(documents)

print(f"Split {len(documents)} document(s) into {len(chunks)} chunks.")
print(f"First chunk (first 200 chars):\n{chunks[0].page_content[:200]}")
print(f"First chunk metadata:\n{chunks[0].metadata}")

Split 1 document(s) into 191 chunks.
First chunk (first 200 chars):
*** START OF THE PROJECT GUTENBERG EBOOK 11 ***

[Illustration]




Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Ho
First chunk metadata:
{'source': 'data/alice_in_wonderland.txt', 'start_index': 0}


**Reasoning**:
Following the instructions, the next step is to initialize an embedding model using `SentenceTransformerEmbeddings` from `langchain_community.embeddings` and generate embeddings for the document chunks. The model 'all-MiniLM-L6-v2' is specified for this purpose.



In [13]:
from langchain_community.embeddings import SentenceTransformerEmbeddings

# Initialize the embedding model
embeddings_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

print("Embedding model initialized successfully.")

/tmp/ipython-input-1724011893.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model initialized successfully.


**Reasoning**:
The embedding model has been successfully initialized. The next step, as per the instructions, is to create a vector store (e.g., using `Chroma` from `langchain_community.vectorstores`) from the generated embeddings and document chunks, and then perform a similarity search using a sample query.



In [14]:
from langchain_community.vectorstores import Chroma

# Create a vector store from the chunks and embeddings
# This will embed the chunks and store them in a ChromaDB instance
vectorstore = Chroma.from_documents(chunks, embeddings_model)

print(f"Vector store created with {vectorstore._collection.count()} items.")

# Perform a similarity search (retrieval)
query = "What is Alice's adventure about?"
docs = vectorstore.similarity_search(query)

print(f"\nRetrieved {len(docs)} documents for the query: '{query}'")
print("--------------------------------------------------")
for i, doc in enumerate(docs):
    print(f"Document {i+1} (Source: {doc.metadata.get('source')}, Start Index: {doc.metadata.get('start_index')}):\n{doc.page_content[:500]}...\n")

Vector store created with 191 items.

Retrieved 4 documents for the query: 'What is Alice's adventure about?'
--------------------------------------------------
Document 1 (Source: data/alice_in_wonderland.txt, Start Index: 83988):
“At any rate I’ll never go _there_ again!” said Alice as she picked her
way through the wood. “It’s the stupidest tea-party I ever was at in
all my life!”

Just as she said this, she noticed that one of the trees had a door
leading right into it. “That’s very curious!” she thought. “But
everything’s curious today. I think I may as well go in at once.” And
in she went.

Once more she found herself in the long hall, and close to the little
glass table. “Now, I’ll manage better this time,” she said...

Document 2 (Source: data/alice_in_wonderland.txt, Start Index: 79779):
“Tell us a story!” said the March Hare.

“Yes, please do!” pleaded Alice.

“And be quick about it,” added the Hatter, “or you’ll be asleep again
before it’s done.”

“Once upon a time there wer

## Generate Lab 2: Vector DB Deep Dive

### Subtask:
Provide the Markdown and Code cells for resource estimation and metadata filtering in Vector Databases.


## Lab 2: Vector DB Deep Dive

This lab delves deeper into the practical aspects of working with Vector Databases, focusing on crucial considerations for building robust and efficient RAG systems. Understanding these concepts is vital for optimizing performance and managing resources effectively.

### Topics Covered:
1.  **Resource Estimation**: How to estimate the memory requirements for storing embeddings in a vector database, based on factors like the number of chunks and embedding dimensions.
2.  **Metadata Filtering**: Enhancing retrieval accuracy and efficiency by leveraging metadata to narrow down search results, allowing for more precise contextual retrieval.

### Resource Estimation for Vector Databases

When building RAG systems, it's crucial to understand the resource implications of storing embeddings. Vector databases store high-dimensional numerical representations of text, and their memory footprint can grow significantly with the volume of data.

Key factors influencing resource estimation:
-   **Number of Documents/Chunks**: Each chunk of text will be converted into a vector. More chunks mean more vectors to store.
-   **Embedding Dimension**: The size of each vector (the number of dimensions) directly impacts memory usage. Larger dimensions lead to larger vectors.
-   **Data Type**: Embeddings are typically stored as floating-point numbers. `float32` (4 bytes per number) is common, but `float16` (2 bytes) can halve memory usage at the cost of some precision, and `float64` (8 bytes) offers higher precision but doubles memory.

To estimate the memory required, you can use the formula: `Number of Chunks * Embedding Dimension * Bytes per float`.

**Reasoning**:
Following the instructions, the next step is to create a Code cell to calculate the estimated memory usage for the existing vector store, using the number of chunks and the embedding dimension with a float32 assumption.



In [15]:
import sys

# Get the number of chunks
num_chunks = len(chunks)

# Determine the embedding dimension
# We'll embed a sample query and get the length of the resulting vector
sample_embedding = embeddings_model.embed_query('test')
embedding_dimension = len(sample_embedding)

# Assume float32 for embedding storage (4 bytes per dimension)
bytes_per_float = 4

# Calculate total estimated memory
estimated_memory_bytes = num_chunks * embedding_dimension * bytes_per_float
estimated_memory_mb = estimated_memory_bytes / (1024**2)
estimated_memory_gb = estimated_memory_bytes / (1024**3)

print(f"Number of chunks: {num_chunks}")
print(f"Embedding dimension: {embedding_dimension}")
print(f"Assumed bytes per float: {bytes_per_float}")
print(f"Estimated memory for embeddings: {estimated_memory_bytes:.2f} bytes")
print(f"Estimated memory for embeddings: {estimated_memory_mb:.2f} MB")
print(f"Estimated memory for embeddings: {estimated_memory_gb:.2f} GB")

Number of chunks: 191
Embedding dimension: 384
Assumed bytes per float: 4
Estimated memory for embeddings: 293376.00 bytes
Estimated memory for embeddings: 0.28 MB
Estimated memory for embeddings: 0.00 GB


### Metadata Filtering

Metadata filtering is a powerful technique to enhance the precision and relevance of retrieval in RAG systems. While vector similarity search is excellent for semantic matching, it doesn't always capture all the nuances or specific requirements of a query.

Metadata refers to additional information or attributes associated with each document or chunk. This can include:
-   **Categorical data**: document type (e.g., 'Introduction', 'Main Body', 'Appendix'), author, publication year, topic.
-   **Numerical data**: `start_index` (position in the original document), page number, section ID.

By leveraging metadata filters, you can:
-   **Narrow down search space**: Restrict similarity search to only chunks that match specific criteria, significantly improving efficiency and relevance.
-   **Improve precision**: Ensure retrieved documents meet specific structural or contextual requirements (e.g., "find answers about Alice's adventures only from the 'Main Body' section").
-   **Enhance control**: Provide more granular control over what information is presented to the language model.

Vector databases like Chroma support robust metadata filtering, allowing you to combine semantic similarity with structured filtering conditions.

**Reasoning**:
Following the instructions, the next step is to create a Code cell to demonstrate metadata filtering. This involves preparing chunks with new metadata, re-creating the vector store, and then performing both numerical and categorical filtered retrievals.



In [16]:
from langchain_core.documents import Document

# a. Prepare Chunks with Metadata
updated_chunks = []
for i, chunk in enumerate(chunks):
    new_metadata = chunk.metadata.copy()
    if i < 10:
        new_metadata['doc_part'] = 'Introduction'
    elif i < 100:
        new_metadata['doc_part'] = 'Main Body'
    else:
        new_metadata['doc_part'] = 'Appendix'
    updated_chunks.append(Document(page_content=chunk.page_content, metadata=new_metadata))

print(f"Prepared {len(updated_chunks)} chunks with updated metadata.")
print(f"Example of updated metadata for first chunk: {updated_chunks[0].metadata}")
print(f"Example of updated metadata for chunk 15: {updated_chunks[15].metadata}")
print(f"Example of updated metadata for chunk 105: {updated_chunks[105].metadata}")

# b. Re-create Vector Store with updated chunks and embeddings
# We'll use a new Chroma instance to demonstrate the filtering clearly
vectorstore_filtered = Chroma.from_documents(updated_chunks, embeddings_model, collection_name="alice_filtered_data")

print(f"Vector store with filtered metadata created with {vectorstore_filtered._collection.count()} items.")

# c. Perform Filtered Retrieval (Numerical Metadata)
query_num = "Who is the main character and what is her story about?"
# Filter for chunks that start after index 50000
docs_num_filtered = vectorstore_filtered.similarity_search(query_num, k=3, where={"start_index": {"$gt": 50000}})

print(f"\nRetrieved {len(docs_num_filtered)} documents for query '{query_num}' with numerical filter (start_index > 50000):")
print("--------------------------------------------------")
for i, doc in enumerate(docs_num_filtered):
    print(f"Document {i+1} (Source: {doc.metadata.get('source')}, Start Index: {doc.metadata.get('start_index')}, Doc Part: {doc.metadata.get('doc_part')}):\n{doc.page_content[:200]}...\n")

# d. Perform Filtered Retrieval (Categorical Metadata)
query_cat = "Who does Alice meet at a tea party?"
# Filter for chunks only from 'Main Body'
docs_cat_filtered = vectorstore_filtered.similarity_search(query_cat, k=3, where={"doc_part": "Main Body"})

print(f"\nRetrieved {len(docs_cat_filtered)} documents for query '{query_cat}' with categorical filter (doc_part = 'Main Body'):")
print("--------------------------------------------------")
for i, doc in enumerate(docs_cat_filtered):
    print(f"Document {i+1} (Source: {doc.metadata.get('source')}, Start Index: {doc.metadata.get('start_index')}, Doc Part: {doc.metadata.get('doc_part')}):\n{doc.page_content[:200]}...\n")

SyntaxError: unterminated f-string literal (detected at line 34) (ipython-input-809439726.py, line 34)